# Notebook 1 — Foundations on a toy database

Welcome to the first hands-on notebook of the MARIO course.

The goal here is to learn the **mechanics** of MARIO without worrying about
heavy downloads or messy real-world data. We use the **built-in test database**
(`mario.load_test(...)`): a tiny, fictitious economy that loads in one line and
behaves exactly like a real one — same methods, same matrices, just small enough
to read with your eyes.

By the end of this notebook you will be able to:

1. **load** and **inspect** an Input-Output Table (IOT) and a Supply-Use Table (SUT);
2. **compute** the derived matrices (coefficients, Leontief inverse, footprints…);
3. **visualize** any matrix with the unified `db.plot(...)` engine;
4. **transform** a SUT into a single-region SUT and into a symmetric **IOT**.

> 💡 The numbers in the test database are made up. Don't read economic meaning
> into them — focus on *what each command does*.

In [1]:
import mario

# Keep the log output short and readable across the whole notebook.
mario.set_log_verbosity("warning")
print("MARIO version:", mario.__version__)

MARIO version: 1.0.2


## Part A — The Input-Output Table (IOT)

### A.1 Load the test table

A MARIO database is an object that bundles together the data matrices, the
labels of every dimension ("sets"), the units of measure and the metadata.
Loading the packaged IOT fixture is a one-liner.

In [2]:
db = mario.load_test("IOT")
db   # evaluating the object prints a compact summary

name = IOT test (standard)
table = IOT
scenarios = ['baseline']
Factor of production = 3
Satellite account = 2
Consumption category = 1
Region = 2
Sector = 3

The summary tells us this is an **IOT**, with a single scenario
(`baseline`), **2 regions** and **3 sectors**, plus factors of production
(value added) and satellite accounts (environmental/social extensions).

### A.2 What is inside? Scenarios, sets and labels

Everything in MARIO is organised by **scenario** and by **set**. A *set* is a
dimension of the table (Region, Sector, Factor of production, …). Let's look at
them.

In [3]:
db.scenarios   # list of scenarios; a fresh database has only 'baseline'

['baseline']

In [4]:
db.sets        # the dimensions of this table

['Factor of production',
 'Satellite account',
 'Consumption category',
 'Region',
 'Sector']

In [5]:
# The labels (items) of every set, in one dictionary
db.get_index("all")

{'Factor of production': ['Taxes', 'Wages', 'Capital'],
 'Satellite account': ['Employment', 'CO2'],
 'Consumption category': ['Final demand'],
 'Region': ['Reg1', 'Reg2'],
 'Sector': ['Agriculture', 'Services', 'Industry']}

In [6]:
# A single set can be accessed by name. Set names are case- and
# singular/plural-insensitive, so all three of these are equivalent.
print(db.get_index("Sector"))
print(db.Sector)
print(db.sectors)

['Agriculture', 'Services', 'Industry']
['Agriculture', 'Services', 'Industry']
['Agriculture', 'Services', 'Industry']


### A.3 Search labels

On the test database you can read all labels at a glance, but real databases
have thousands of sectors. `search(...)` is how you find the label you need.

In [7]:
db.search("Sector", "Agr")

['Agriculture']

### A.4 Look at the stored matrices

MARIO stores only a few "primary" matrices and **computes the rest on demand**.
Let's see what is already available, and then query one matrix.

Naming convention (used everywhere in MARIO):

| Symbol | Meaning |
|---|---|
| `Z` / `z` | intermediate transactions (flows) / technical coefficients |
| `Y` | final demand |
| `X` | total output |
| `V` / `v` | value added (flows / coefficients) |
| `E` / `e` | satellite accounts (flows / coefficients) |
| `w` | Leontief inverse `(I − z)⁻¹` |
| `f` | footprint intensities |

**Uppercase = absolute flows, lowercase = coefficients.**

In [8]:
db.list_matrices(scenario="baseline")   # which matrices are stored right now

('E', 'EY', 'V', 'VY', 'Y', 'Z')

In [9]:
# Z: how much each sector sells to each sector (intermediate transactions)
db.Z

Region                             Reg1                              \
Level                            Sector                               
Item                        Agriculture      Industry      Services   
Region Level  Item                                                    
Reg1   Sector Agriculture  9.308651e+05  5.357799e+06  1.075278e+06   
              Industry     1.122397e+06  2.289284e+07  1.065096e+07   
              Services     1.893586e+06  9.737509e+06  3.059785e+07   
Reg2   Sector Agriculture  4.168587e+02  2.596041e+03  8.430071e+02   
              Industry     6.890987e+03  1.335193e+05  6.555829e+04   
              Services     1.590090e+03  1.555045e+04  3.843707e+04   

Region                             Reg2                                
Level                            Sector                                
Item                        Agriculture       Industry       Services  
Region Level  Item                                                     
Reg1   Sector Agriculture   2363.029548   50607.806861   10341.916110  
              Industry      1413.697219  134910.657543   30367.209709  
              Services      1301.725200   27813.857216   58525.384254  
Reg2   Sector Agriculture   2962.557738   30635.045696    7961.643900  
              Industry      9736.931972  287300.673355  161126.940614  
              Services     16680.335207  312355.775242  722089.042972

`query(...)` retrieves one or more matrices and, crucially, **computes them
if they are missing**. Below we ask for the satellite-account coefficients `e`
(emissions/employment per unit of output): MARIO resolves them automatically
from `E` and `X`.

In [10]:
db.query("e")

Region               Reg1                                         Reg2  \
Level              Sector                                       Sector   
Item          Agriculture       Industry       Services    Agriculture   
Item                                                                     
Employment       0.170734       0.012327       0.019621       0.009259   
CO2         342552.270192  167269.943085  179791.790027  289962.466193   

Region                                  
Level                                   
Item            Industry      Services  
Item                                    
Employment      0.003295      0.007659  
CO2         40474.898330  69431.794242

## Part B — Computing matrices

The heart of MARIO. `calc_all([...])` computes a list of target matrices and
**resolves all dependencies internally** (e.g. `w` needs `z`, which needs `Z`
and `X`). Results are then plain `pandas.DataFrame` accessible as attributes.

In [11]:
db.calc_all(["X", "z", "w", "f"])
db.list_matrices()

('E', 'EY', 'V', 'VY', 'X', 'Y', 'Z', 'e', 'f', 'w', 'z')

In [12]:
db.X    # total output by region and sector

production
Region Level  Item                     
Reg1   Sector Agriculture  9.972169e+06
              Industry     5.311550e+07
              Services     1.044952e+08
Reg2   Sector Agriculture  7.984966e+04
              Industry     1.195683e+06
              Services     2.418662e+06

In [13]:
db.z    # technical coefficients: input per unit of output

Region                           Reg1                            Reg2  \
Level                          Sector                          Sector   
Item                      Agriculture  Industry  Services Agriculture   
Region Level  Item                                                      
Reg1   Sector Agriculture    0.093346  0.100871  0.010290    0.029593   
              Industry       0.112553  0.431001  0.101928    0.017704   
              Services       0.189887  0.183327  0.292816    0.016302   
Reg2   Sector Agriculture    0.000042  0.000049  0.000008    0.037102   
              Industry       0.000691  0.002514  0.000627    0.121941   
              Services       0.000159  0.000293  0.000368    0.208897   

Region                                         
Level                                          
Item                       Industry  Services  
Region Level  Item                             
Reg1   Sector Agriculture  0.042325  0.004276  
              Industry     0.112831  0.012555  
              Services     0.023262  0.024197  
Reg2   Sector Agriculture  0.025621  0.003292  
              Industry     0.240282  0.066618  
              Services     0.261236  0.298549

`f` are the **footprint intensities**: the total (direct + indirect) amount
of each satellite account embodied in one unit of final demand of each sector.
This is already a small "impact" model.

In [14]:
db.f

Region               Reg1                                         Reg2  \
Level              Sector                                       Sector   
Item          Agriculture       Industry       Services    Agriculture   
Item                                                                     
Employment       0.205834       0.071561       0.041096       0.026342   
CO2         508912.512077  492572.973155  332919.806550  392045.986026   

Region                                    
Level                                     
Item             Industry       Services  
Item                                      
Employment       0.034874       0.018308  
CO2         228374.179654  145915.355204

### B.1 GDP and other indicators

Beyond raw matrices, MARIO exposes high-level indicators. `GDP()` sums value
added by region.

In [15]:
db.GDP()

,GDP
Region,
Reg1,8.305835e+07
Reg2,1.825701e+06


## Part C — Visualization

MARIO ships a single, unified plotting engine: `db.plot(...)`, built on Plotly.
The quickest way in is to pass a matrix name and a **preset**; for full control
you can map columns explicitly (`x`, `color`, `facet_col`, `top_n`, `filters`…).

> In these cells we use `path=False, auto_open=False` so the figure is returned
> inline instead of being written to an HTML file.

In [16]:
# Quick look: let MARIO pick a sensible mapping for Z
fig = db.plot(matrix="Z", preset="overview", path=False, auto_open=False)
fig

In [17]:
# Explicit mapping: final demand Y, one bar group per sector, colored by region
fig = db.plot(
    matrix="Y",
    kind="bar",
    preset=None,
    x="Sector_from",
    color="Region_to",
    agg="sum",
    path=False,
    auto_open=False,
)
fig

In [18]:
# Set return_data=True to also get the dataframe that was actually plotted
fig, plotted = db.plot(
    matrix="Z",
    preset=None,
    kind="bar",
    x="Sector_from",
    color="Region_to",
    filters={"Region_from": ["Reg1"]},   # keep only flows originating in Reg1
    path=False,
    auto_open=False,
    return_data=True,
)
plotted.head()

,Sector_from,Region_to,Value
0,Agriculture,Reg1,7.363942e+06
1,Agriculture,Reg2,6.331275e+04
2,Industry,Reg1,3.466619e+07
3,Industry,Reg2,1.666916e+05
4,Services,Reg1,4.222895e+07


## Part D — The same workflow on a SUT

A **Supply-Use Table (SUT)** splits the economy into **Activities** (industries)
and **Commodities** (products), with two core blocks: **Supply** `S` (who makes
what) and **Use** `U` (who uses what). All the inspection and computation methods
you just learned work identically — you just get the activity/commodity blocks.

In [19]:
sut = mario.load_test("SUT")
sut

name = SUT test (standard)
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 2
Commodity = 2
Factor of production = 3
Satellite account = 2
Consumption category = 1
Region = 2

In [20]:
sut.get_index("all")

{'Activity': ['Manufacturing', 'Services'],
 'Commodity': ['Goods', 'Services'],
 'Factor of production': ['Taxes', 'Wages', 'Capital'],
 'Satellite account': ['Employment', 'CO2'],
 'Consumption category': ['Final demand'],
 'Region': ['Region 1', 'Region 2']}

In [21]:
# The Use matrix: commodities used by each activity
sut.U

Region                           Region 1                    Region 2  \
Level                            Activity                    Activity   
Item                        Manufacturing      Services Manufacturing   
Region   Level     Item                                                 
Region 1 Commodity Goods     3.334903e+11  1.657755e+11  1.487965e+11   
                   Services  3.261811e+11  7.254021e+11  1.176727e+10   
Region 2 Commodity Goods     1.922623e+11  4.099986e+10  3.038628e+13   
                   Services  2.614852e+10  5.823465e+10  1.154871e+13   

Region                                     
Level                                      
Item                             Services  
Region   Level     Item                    
Region 1 Commodity Goods     6.895332e+10  
                   Services  3.588505e+10  
Region 2 Commodity Goods     1.157115e+13  
                   Services  3.075294e+13

In [22]:
# Compute on a SUT exactly as on an IOT
sut.calc_all(["Xc", "u"])
sut.u    # use coefficients

Region                           Region 1                Region 2          
Level                            Activity                Activity          
Item                        Manufacturing  Services Manufacturing  Services
Region   Level     Item                                                    
Region 1 Commodity Goods         0.261452  0.068540      0.002359  0.000660
                   Services      0.255721  0.299919      0.000187  0.000343
Region 2 Commodity Goods         0.150731  0.016951      0.481652  0.110734
                   Services      0.020500  0.024077      0.183058  0.294300

SUT-specific plots work the same way — here the use of commodities by
activity, one panel per region.

In [23]:
fig = sut.plot(
    matrix="U",
    kind="bar",
    preset=None,
    x="Commodity_from",
    color="Activity_to",
    facet_col="Region_to",
    path=False,
    auto_open=False,
)
fig

## Part E — Transforming a SUT

This is the new material for the SUT module, and the reason SUTs matter in
practice: they are the **raw form** in which statistical offices publish data,
but most analysis is done on symmetric **IOTs**. MARIO bridges the two.

### E.1 SUT → single-region SUT (MRIO → SRIO)

A multi-regional table can be collapsed onto **one region of interest**, with
all the other regions treated as "rest of the world" (imports/exports). Use
`to_single_region(...)`. We work on a copy (`inplace=False`) to keep `sut`
intact.

In [24]:
sut_multi = mario.load_test("SUT")
print("Regions before:", sut_multi.regions)

sut_region1 = sut_multi.to_single_region("Region 1", inplace=False)
print("Regions after :", sut_region1.regions)
sut_region1

Regions before: ['Region 1', 'Region 2']
WARNING All the scenarios will be deleted to build up the new baseline.


Regions after : ['Region 1']


name = SUT test (standard)
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 2
Commodity = 2
Factor of production = 4
Satellite account = 2
Consumption category = 3
Region = 1

### E.2 SUT → IOT

The key transformation. `to_iot(method)` builds a **symmetric Input-Output
Table** from the supply and use blocks. The `method` argument selects the
construction assumption:

| Method | Assumption |
|---|---|
| `"A"` | product technology, commodity-by-commodity |
| `"B"` | industry technology, commodity-by-commodity |
| `"C"` | fixed product sales structure, industry-by-industry |
| `"D"` | fixed industry sales structure, industry-by-industry |

We use `"B"` (industry-technology assumption), one of the most common choices.
After the conversion the table type flips from `SUT` to `IOT`, and the
`Activity`/`Commodity` sets are replaced by a single `Sector` set.

In [25]:
sut_to_convert = mario.load_test("SUT")
print("Before:", sut_to_convert.table_type, "| sets:", sut_to_convert.sets)

# Convert in place (the default). Pass inplace=False to keep the SUT as well.
sut_to_convert.to_iot("B")

print("After :", sut_to_convert.table_type, "| sets:", sut_to_convert.sets)
sut_to_convert

Before: SUT | sets: ['Activity', 'Commodity', 'Factor of production', 'Satellite account', 'Consumption category', 'Region']


WARNING baseline deleted from the database


After : IOT | sets: ['Factor of production', 'Satellite account', 'Consumption category', 'Region', 'Sector']


name = SUT test (standard)
table = IOT
scenarios = ['baseline']
Factor of production = 3
Satellite account = 2
Consumption category = 1
Region = 2
Sector = 2

In [26]:
# It is now a normal IOT: we can compute and inspect it like any other.
sut_to_convert.calc_all(["X", "z"])
sut_to_convert.z

Region                    Region 1            Region 2          
Level                       Sector              Sector          
Item                         Goods  Services     Goods  Services
Region   Level  Item                                            
Region 1 Sector Goods     0.251041  0.071364  0.002343  0.000670
                Services  0.258107  0.299272  0.000188  0.000342
Region 2 Sector Goods     0.143511  0.018910  0.478175  0.113042
                Services  0.020693  0.024025  0.184101  0.293608

### 🧩 Your turn

Try the following on a fresh database (use the cells above as a reference):

1. Load a new SUT (`mario.load_test("SUT")`).
2. Convert it to an IOT with a **different** method (e.g. `"D"`).
3. Compute the Leontief inverse `w` and print its shape.

Then move on to **Notebook 2**, where we leave the toy data behind and parse a
real, multi-regional database (EXIOBASE).

In [27]:
# 🧩 Exercise — write your code here
